<a href="https://colab.research.google.com/github/rifahalam123/Build-A-Game-Challenge-INNOVATEX-4-Presidency-University/blob/main/spark_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EXPERIMENT-1

In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("FraudFeatureEngineering") \
    .master("local[*]") \
    .getOrCreate()

data = [
    (1, "C101", 1000, "2026-11-01 10:00:00", "Bangalore", "D1"),
    (2, "C101", 1200, "2026-11-01 10:05:00", "Bangalore", "D1"),
    (3, "C101", 15000, "2026-11-01 10:10:00", "Mumbai", "D2"),
    (4, "C102", 500, "2026-11-01 14:00:00", "Delhi", "D3"),
    (5, "C102", 700, "2026-11-01 14:10:00", "Delhi", "D3"),
    (6, "C103", 25000, "2026-11-01 02:30:00", "Chennai", "D4"),
    (7, "C103", 30000, "2026-11-01 02:45:00", "Kolkata", "D5"),
    (8, "C104", 800, "2026-11-01 11:00:00", "Hyderabad", "D6"),
    (9, "C104", 900, "2026-11-01 11:05:00", "Hyderabad", "D7"),
    (10, "C104", 10000, "2026-11-01 23:30:00", "Pune", "D8")
]

columns = [
    "transaction_id", "customer_id", "amount",
    "transaction_time", "location", "device_id"
]

df = spark.createDataFrame(data, columns)

df = df.withColumn(
    "transaction_time",
    to_timestamp("transaction_time")
)

print("ORIGINAL TRANSACTION DATA")
df.show(truncate=False)

# 1. Transaction Velocity
w = Window.partitionBy("customer_id").orderBy("transaction_time")

df = df.withColumn(
    "transaction_velocity",
    count("transaction_id").over(
        w.rowsBetween(-2, 0)
    )
)

# 2. Amount Deviation
customer_window = Window.partitionBy("customer_id")

df = df.withColumn(
    "average_amount",
    avg("amount").over(customer_window)
)

df = df.withColumn(
    "amount_deviation",
    abs(col("amount") - col("average_amount"))
)

# 3. Geographical Distance (simplified location-change indicator)
df = df.withColumn(
    "previous_location",
    lag("location").over(w)
)

df = df.withColumn(
    "location_change",
    when(col("previous_location").isNull(), 0)
    .when(col("location") != col("previous_location"), 1)
    .otherwise(0)
)

# 4. Unusual Hour Activity
df = df.withColumn(
    "transaction_hour",
    hour("transaction_time")
)

df = df.withColumn(
    "unusual_hour",
    when(
        (col("transaction_hour") < 6) |
        (col("transaction_hour") >= 23), 1
    ).otherwise(0)
)

# 5. Device Frequency
df = df.withColumn(
    "device_frequency",
    count("transaction_id").over(
        Window.partitionBy("customer_id", "device_id")
    )
)

# 6. Customer Behavioral Features
df = df.withColumn(
    "customer_transaction_count",
    count("transaction_id").over(customer_window)
)

df = df.withColumn(
    "customer_total_amount",
    sum("amount").over(customer_window)
)

df = df.withColumn(
    "high_amount_flag",
    when(col("amount") > col("average_amount") * 2, 1)
    .otherwise(0)
)

# 7. Fraud Risk Score
df = df.withColumn(
    "fraud_risk_score",
    col("location_change") +
    col("unusual_hour") +
    col("high_amount_flag") +
    when(col("transaction_velocity") >= 3, 1).otherwise(0)
)

df = df.withColumn(
    "fraud_status",
    when(col("fraud_risk_score") >= 2, "Suspicious")
    .otherwise("Normal")
)

print("FRAUD FEATURE ENGINEERING OUTPUT")

df.select(
    "transaction_id",
    "customer_id",
    "amount",
    "transaction_velocity",
    "amount_deviation",
    "location_change",
    "unusual_hour",
    "device_frequency",
    "customer_transaction_count",
    "customer_total_amount",
    "high_amount_flag",
    "fraud_risk_score",
    "fraud_status"
).show(truncate=False)

print("SUSPICIOUS TRANSACTIONS")

df.filter(
    col("fraud_status") == "Suspicious"
).select(
    "transaction_id",
    "customer_id",
    "amount",
    "fraud_risk_score",
    "fraud_status"
).show()

spark.stop()

ORIGINAL TRANSACTION DATA
+--------------+-----------+------+-------------------+---------+---------+
|transaction_id|customer_id|amount|transaction_time   |location |device_id|
+--------------+-----------+------+-------------------+---------+---------+
|1             |C101       |1000  |2026-11-01 10:00:00|Bangalore|D1       |
|2             |C101       |1200  |2026-11-01 10:05:00|Bangalore|D1       |
|3             |C101       |15000 |2026-11-01 10:10:00|Mumbai   |D2       |
|4             |C102       |500   |2026-11-01 14:00:00|Delhi    |D3       |
|5             |C102       |700   |2026-11-01 14:10:00|Delhi    |D3       |
|6             |C103       |25000 |2026-11-01 02:30:00|Chennai  |D4       |
|7             |C103       |30000 |2026-11-01 02:45:00|Kolkata  |D5       |
|8             |C104       |800   |2026-11-01 11:00:00|Hyderabad|D6       |
|9             |C104       |900   |2026-11-01 11:05:00|Hyderabad|D7       |
|10            |C104       |10000 |2026-11-01 23:30:00|Pune   

EXPERIMENT - 2


In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("HealthcareBigDataAnalysis") \
    .master("local[*]") \
    .getOrCreate()

data = [
    (1, "Amit", 45, "Diabetes", 180, 120, 5000),
    (2, "Riya", 30, "Heart Disease", 150, 95, 12000),
    (3, "Rahul", 60, "Diabetes", 200, 140, 8000),
    (4, "Priya", 25, "Asthma", 110, 75, 3000),
    (5, "Karan", 55, "Heart Disease", 190, 130, 15000),
    (6, "Neha", 40, "Asthma", 120, 80, 4000),
    (7, "Arjun", 65, "Diabetes", 210, 150, 10000),
    (8, "Sneha", 35, "Heart Disease", 160, 100, 9000),
    (9, "Vikram", 50, "Diabetes", 175, 115, 7000),
    (10, "Anjali", 28, "Asthma", 105, 70, 2500)
]

columns = [
    "patient_id", "patient_name", "age",
    "disease", "systolic_bp", "diastolic_bp",
    "treatment_cost"
]

df = spark.createDataFrame(data, columns)

print("HEALTHCARE PATIENT DATA")
df.show()

# 1. Total Number of Patients
print("TOTAL NUMBER OF PATIENTS")
df.select(count("patient_id").alias("Total_Patients")).show()

# 2. Disease Analysis
print("DISEASE-WISE PATIENT COUNT")
df.groupBy("disease").count().show()

# 3. Average Treatment Cost
print("AVERAGE TREATMENT COST BY DISEASE")
df.groupBy("disease").agg(
    round(avg("treatment_cost"), 2).alias("Average_Cost")
).show()

# 4. Age Group Analysis
print("AGE GROUP ANALYSIS")
df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Middle Age")
    .otherwise("Senior")
).groupBy("age_group").count().show()

# 5. High Blood Pressure Analysis
print("PATIENTS WITH HIGH BLOOD PRESSURE")
df.filter(
    (col("systolic_bp") >= 140) |
    (col("diastolic_bp") >= 90)
).select(
    "patient_name", "age", "systolic_bp", "diastolic_bp"
).show()

# 6. Most Common Disease
print("MOST COMMON DISEASE")
df.groupBy("disease").count().orderBy(
    desc("count")
).limit(1).show()

# 7. Total Healthcare Treatment Cost
print("TOTAL TREATMENT COST")
df.agg(
    sum("treatment_cost").alias("Total_Cost")
).show()

# 8. High Treatment Cost Patients
print("PATIENTS WITH HIGH TREATMENT COST")
df.filter(
    col("treatment_cost") >= 10000
).select(
    "patient_name", "disease", "treatment_cost"
).show()

print("HEALTHCARE BIG DATA ANALYSIS COMPLETED")

spark.stop()

HEALTHCARE PATIENT DATA
+----------+------------+---+-------------+-----------+------------+--------------+
|patient_id|patient_name|age|      disease|systolic_bp|diastolic_bp|treatment_cost|
+----------+------------+---+-------------+-----------+------------+--------------+
|         1|        Amit| 45|     Diabetes|        180|         120|          5000|
|         2|        Riya| 30|Heart Disease|        150|          95|         12000|
|         3|       Rahul| 60|     Diabetes|        200|         140|          8000|
|         4|       Priya| 25|       Asthma|        110|          75|          3000|
|         5|       Karan| 55|Heart Disease|        190|         130|         15000|
|         6|        Neha| 40|       Asthma|        120|          80|          4000|
|         7|       Arjun| 65|     Diabetes|        210|         150|         10000|
|         8|       Sneha| 35|Heart Disease|        160|         100|          9000|
|         9|      Vikram| 50|     Diabetes|        1

EXPERIMENT - 3

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, variance

# Note: We reuse the active Spark session and the pre-existing Spark DataFrame `df`
# representing a high-dimensional dataset in the kernel state.

# 1. Inspect Initial Columns
initial_cols = df.columns
num_initial_cols = len(initial_cols)
print(f"Initial Features ({num_initial_cols}): {initial_cols}\n")

# 2. Calculate Feature Variances
print("--- STEP 1: VARIANCE ANALYSIS ---")
variance_threshold = 0.01
low_variance_cols = []
constant_cols = []

for col_name in initial_cols:
    # Calculate variance
    var_val = df.select(variance(col(col_name))).collect()[0][0]
    var_val = var_val if var_val is not None else 0.0
    print(f"Feature: {col_name:<22} | Variance: {var_val:.8f}")

    if var_val == 0.0:
        constant_cols.append(col_name)
    elif var_val < variance_threshold:
        low_variance_cols.append(col_name)

print(f"\nConstant Columns: {constant_cols}")
print(f"Low-Variance Columns (Threshold < {variance_threshold}): {low_variance_cols}")

Initial Features (10): ['Transaction_Amount', 'Amount_Duplicate', 'Account_Balance', 'Balance_Duplicate', 'Customer_Age', 'Transaction_Count', 'Device_Risk', 'Low_Variance_Feature', 'Constant_Feature', 'Random_Feature']

--- STEP 1: VARIANCE ANALYSIS ---
Feature: Transaction_Amount     | Variance: 9770.64356928
Feature: Amount_Duplicate       | Variance: 9773.38343523
Feature: Account_Balance        | Variance: 1010736.76828142
Feature: Balance_Duplicate      | Variance: 1010582.42123022
Feature: Customer_Age           | Variance: 182.72130440
Feature: Transaction_Count      | Variance: 30.42980590
Feature: Device_Risk            | Variance: 0.08653818
Feature: Low_Variance_Feature   | Variance: 0.00000001
Feature: Constant_Feature       | Variance: 0.00000000
Feature: Random_Feature         | Variance: 0.97587910

Constant Columns: ['Constant_Feature']
Low-Variance Columns (Threshold < 0.01): ['Low_Variance_Feature']


In [17]:
import builtins

# 3. Calculate Pairwise Correlation
print("--- STEP 2: PAIRWISE CORRELATION ANALYSIS ---")
correlation_threshold = 0.95
highly_correlated_pairs = []

# Exclude constant columns first to prevent division by zero in correlation calculations
candidate_cols = [c for c in initial_cols if c not in constant_cols]

for i in range(len(candidate_cols)):
    for j in range(i + 1, len(candidate_cols)):
        col1 = candidate_cols[i]
        col2 = candidate_cols[j]
        corr_val = df.stat.corr(col1, col2)
        if builtins.abs(corr_val) >= correlation_threshold:
            highly_correlated_pairs.append((col1, col2, corr_val))
            print(f"Redundant Pair: ({col1}, {col2}) | Correlation: {corr_val:.6f}")

--- STEP 2: PAIRWISE CORRELATION ANALYSIS ---
Redundant Pair: (Transaction_Amount, Amount_Duplicate) | Correlation: 0.999798
Redundant Pair: (Account_Balance, Balance_Duplicate) | Correlation: 0.999988


In [18]:
# 4. Rule-Based Elimination & Evaluation
print("--- STEP 3: RULE-BASED SELECTION & EVALUATION ---")

# Rules:
# 1. Drop all constant and low-variance features.
# 2. For highly correlated pairs, drop the second column to avoid multi-collinearity.
drop_cols = set(constant_cols + low_variance_cols)

for col1, col2, corr in highly_correlated_pairs:
    if col1 not in drop_cols and col2 not in drop_cols:
        # Drop col2 as it's redundant to col1
        drop_cols.add(col2)

print(f"Columns to Drop: {list(drop_cols)}")

# Drop columns and evaluate remaining dimensionality
filtered_df = df.drop(*drop_cols)
final_cols = filtered_df.columns
num_final_cols = len(final_cols)

print(f"\nDimensionality Summary:")
print(f"- Initial Features: {num_initial_cols}")
print(f"- Final Features:    {num_final_cols}")
print(f"- Kept Features:     {final_cols}")
print(f"- Reduction:         {((num_initial_cols - num_final_cols) / num_initial_cols) * 100:.2f}%")

# Display sample outputs of the filtered dataset
filtered_df.show(5, truncate=False)

--- STEP 3: RULE-BASED SELECTION & EVALUATION ---
Columns to Drop: ['Amount_Duplicate', 'Constant_Feature', 'Balance_Duplicate', 'Low_Variance_Feature']

Dimensionality Summary:
- Initial Features: 10
- Final Features:    6
- Kept Features:     ['Transaction_Amount', 'Account_Balance', 'Customer_Age', 'Transaction_Count', 'Device_Risk', 'Random_Feature']
- Reduction:         40.00%
+------------------+------------------+------------+-----------------+-------------------+--------------------+
|Transaction_Amount|Account_Balance   |Customer_Age|Transaction_Count|Device_Risk        |Random_Feature      |
+------------------+------------------+------------+-----------------+-------------------+--------------------+
|549.6714153011233 |4324.821725025618 |25.0        |6.0              |0.6141862443190114 |-1.595636760889683  |
|486.1735698828815 |4855.481329284482 |49.0        |12.0             |0.23981995220659025|-1.4055630538562867 |
|564.7688538100692 |4207.5800790000385|51.0        |8.0

EXPERIMENT - 4

### Task: Dependency Graph Analysis using PySpark
In this experiment, we construct a directed graph representing dependencies among software packages. We will:
1. Initialize a Spark Session supporting graph processing.
2. Define vertices (packages) and directed edges (dependencies).
3. Compute vertex degrees (in-degree, out-degree) to identify dependency bottlenecks.
4. Apply BFS to determine the dependency path/chain between two packages.
5. Simulate the removal of a critical node to observe the cascading impact on the network structure.

In [31]:
import networkx as nx
import pandas as pd

print("NetworkX and Pandas successfully imported for graph analysis!")

NetworkX and Pandas successfully imported for graph analysis!


In [32]:
# 1. Construct Directed Graph representing dependencies
dependency_graph = nx.DiGraph()

# Define vertices (Software Packages)
packages = [
    ("P1", {"name": "Core_Auth", "type": "Library"}),
    ("P2", {"name": "Database_Connector", "type": "Library"}),
    ("P3", {"name": "Logging_Utility", "type": "Utility"}),
    ("P4", {"name": "Payment_Gateway", "type": "Service"}),
    ("P5", {"name": "User_Dashboard", "type": "UI_Component"}),
    ("P6", {"name": "Notification_Engine", "type": "Service"}),
    ("P7", {"name": "Analytics_Suite", "type": "Service"}),
    ("P8", {"name": "External_API_Adapter", "type": "Library"})
]
dependency_graph.add_nodes_from(packages)

# Define directed edges (src depends on dst)
dependencies = [
    ("P5", "P1"), # User_Dashboard -> Core_Auth
    ("P5", "P2"), # User_Dashboard -> Database_Connector
    ("P4", "P1"), # Payment_Gateway -> Core_Auth
    ("P4", "P2"), # Payment_Gateway -> Database_Connector
    ("P1", "P3"), # Core_Auth -> Logging_Utility
    ("P2", "P3"), # Database_Connector -> Logging_Utility
    ("P6", "P1"), # Notification_Engine -> Core_Auth
    ("P7", "P2"), # Analytics_Suite -> Database_Connector
    ("P7", "P8"), # Analytics_Suite -> External_API_Adapter
    ("P8", "P3")  # External_API_Adapter -> Logging_Utility
]
dependency_graph.add_edges_from(dependencies)

print(f"Dependency Graph Constructed Successfully with {dependency_graph.number_of_nodes()} nodes and {dependency_graph.number_of_edges()} edges!")

Dependency Graph Constructed Successfully with 8 nodes and 10 edges!


In [33]:
print("--- VERTEX DEGREE & BOTTLENECK ANALYSIS ---")
# In-degree measures how many times a node is a dependency of other nodes (crucial bottlenecks)
in_degrees = dict(dependency_graph.in_degree())
# Out-degree measures how many external packages this node depends on
out_degrees = dict(dependency_graph.out_degree())

# Build a DataFrame of node profiles
node_profiles = []
for node in dependency_graph.nodes:
    node_profiles.append({
        "id": node,
        "name": dependency_graph.nodes[node]["name"],
        "type": dependency_graph.nodes[node]["type"],
        "in_degree_dependents": in_degrees[node],
        "out_degree_dependencies": out_degrees[node]
    })

df_profiles = pd.DataFrame(node_profiles)
print("\nPackage Dependency Profiles (Sorted by In-Degree Bottlenecks):")
print(df_profiles.sort_values(by="in_degree_dependents", ascending=False).to_string(index=False))

--- VERTEX DEGREE & BOTTLENECK ANALYSIS ---

Package Dependency Profiles (Sorted by In-Degree Bottlenecks):
id                 name         type  in_degree_dependents  out_degree_dependencies
P1            Core_Auth      Library                     3                        1
P2   Database_Connector      Library                     3                        1
P3      Logging_Utility      Utility                     3                        0
P8 External_API_Adapter      Library                     1                        1
P4      Payment_Gateway      Service                     0                        2
P5       User_Dashboard UI_Component                     0                        2
P6  Notification_Engine      Service                     0                        1
P7      Analytics_Suite      Service                     0                        2


In [34]:
print("--- PATHWAY ANALYSIS (BFS/Shortest Paths) ---")
# Find dependency chains from User_Dashboard (P5) to Logging Utility (P3)
source = "P5"
target = "P3"

try:
    # Get all simple dependency paths from source to target
    paths = list(nx.all_simple_paths(dependency_graph, source=source, target=target))
    print(f"Dependency paths from {dependency_graph.nodes[source]['name']} ({source}) to {dependency_graph.nodes[target]['name']} ({target}):")
    for idx, path in enumerate(paths, 1):
        path_names = [f"{dependency_graph.nodes[node]['name']} ({node})" for node in path]
        print(f"  Path {idx}: {' -> '.join(path_names)}")
except nx.NetworkXNoPath:
    print(f"No dependency path exists between {source} and {target}.")

--- PATHWAY ANALYSIS (BFS/Shortest Paths) ---
Dependency paths from User_Dashboard (P5) to Logging_Utility (P3):
  Path 1: User_Dashboard (P5) -> Core_Auth (P1) -> Logging_Utility (P3)
  Path 2: User_Dashboard (P5) -> Database_Connector (P2) -> Logging_Utility (P3)


In [35]:
print("--- IMPACT ANALYSIS: REMOVING CRITICAL NODE (P1: Core_Auth) ---")
critical_node = "P1"

# Copy graph before simulation
simulated_graph = dependency_graph.copy()

# Calculate original graph connectivity attributes
print(f"Original Node Count: {simulated_graph.number_of_nodes()}")
print(f"Original Edge Count: {simulated_graph.number_of_edges()}")

# Remove the critical node representing service outage or package deprecation
simulated_graph.remove_node(critical_node)

print(f"\nSimulating removal of node: {dependency_graph.nodes[critical_node]['name']} ({critical_node})")
print(f"Post-Removal Node Count: {simulated_graph.number_of_nodes()}")
print(f"Post-Removal Edge Count: {simulated_graph.number_of_edges()}")

# Show disconnected sub-graphs or isolated elements
isolated = list(nx.isolates(simulated_graph))
isolated_names = [f"{simulated_graph.nodes[node]['name']} ({node})" for node in isolated]
print(f"Isolated packages left with zero active dependencies: {isolated_names}")

--- IMPACT ANALYSIS: REMOVING CRITICAL NODE (P1: Core_Auth) ---
Original Node Count: 8
Original Edge Count: 10

Simulating removal of node: Core_Auth (P1)
Post-Removal Node Count: 7
Post-Removal Edge Count: 6
Isolated packages left with zero active dependencies: ['Notification_Engine (P6)']


EXPERIMENT - 5


### Experiment 5: Predictive Maintenance Data Science Project

In this experiment, we will build a PySpark-based data science pipeline to predict industrial machine failure. The workflow includes:
1. **Data Definition**: Generating synthetic telemetry data with features like `temperature`, `vibration`, `pressure`, and `operational_hours` for multiple machines.
2. **Feature Engineering**: Creating rolling statistics (window-based moving averages and standard deviations) to capture degradation over time.
3. **Label Definition**: Defining a failure target label based on critical thresholds.
4. **Data Preparation & Summary**: Aggregating features for training and analyzing predictive signals.

In [42]:
import pandas as pd
import numpy as np
import datetime
import random
import builtins

# Generate synthetic industrial sensor data using Pandas
machines = ["M_1001", "M_1002", "M_1003", "M_1004", "M_1005"]
base_time = datetime.datetime(2026, 11, 1, 0, 0, 0)

raw_data = []
random.seed(42)

record_id = 1
for m in machines:
    for hour in range(120):
        timestamp = base_time + datetime.timedelta(hours=hour)
        # Normal operational noise with occasional spike conditions towards the end
        deg_factor = (hour / 120.0) ** 2 if m in ["M_1001", "M_1003"] else 0.0
        temp = builtins.round(65.0 + (random.normalvariate(0, 1) * 3) + (deg_factor * 25.0), 2)
        vibe = builtins.round(2.5 + (random.normalvariate(0, 1) * 0.4) + (deg_factor * 3.5), 2)
        press = builtins.round(100.0 + (random.normalvariate(0, 1) * 8) + (deg_factor * 15.0), 2)
        op_hours = hour + 1

        raw_data.append((record_id, m, timestamp, temp, vibe, press, op_hours))
        record_id += 1

columns = ["record_id", "machine_id", "timestamp", "temperature", "vibration", "pressure", "operational_hours"]
transient_df = pd.DataFrame(raw_data, columns=columns)

print("RAW INDUSTRIAL TELEMETRY DATA SAMPLE (PANDAS):")
print(transient_df.head(5).to_string(index=False))

RAW INDUSTRIAL TELEMETRY DATA SAMPLE (PANDAS):
 record_id machine_id           timestamp  temperature  vibration  pressure  operational_hours
         1     M_1001 2026-11-01 00:00:00        65.74       2.30    110.04                  1
         2     M_1001 2026-11-01 01:00:00        64.59       2.11    104.52                  2
         3     M_1001 2026-11-01 02:00:00        61.50       3.20     97.41                  3
         4     M_1001 2026-11-01 03:00:00        68.56       3.10    115.61                  4
         5     M_1001 2026-11-01 04:00:00        71.96       2.32    111.87                  5
